In [ ]:
"""
================================================================================
Sentiment Analysis - Amazon Fine Food Reviews
CodeAlpha Data Analytics Internship - Task 3
================================================================================
Goal: classify reviews as Positive/Negative/Neutral, detect specific emotions,
surface opinion trends, and translate results into actionable business insight.

Pipeline:
 1. Load, deduplicate, and sample the data
 2. Clean review text
 3. Classify sentiment with VADER (lexicon + rule-based NLP)
 4. Validate VADER labels against the star-rating ground truth
 5. Detect 8 discrete emotions with the NRC Emotion Lexicon
 6. Analyze trends over time and by helpfulness
 7. Visualize results + export labeled dataset
================================================================================
"""
import pandas as pd
import numpy as np
import re
import html
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install nrclex
from nrclex import NRCLex
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RANDOM_SEED = 42
SAMPLE_SIZE = 30000
DATA_PATH = 'Reviews.csv'          # https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
OUT_DIR = 'charts'
EMOTIONS = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'disgust', 'joy']
ORDER = ['Positive', 'Neutral', 'Negative']
COLORS = {'Positive': '#2ecc71', 'Neutral': '#95a5a6', 'Negative': '#e74c3c'}

import os
os.makedirs(OUT_DIR, exist_ok=True)
sns.set_style('whitegrid')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.5 MB/s eta 0:00:00


In [ ]:
# ------------------------------------------------------------------ #
# 1. Load, deduplicate, sample
# ------------------------------------------------------------------ #
print("Loading data...")
df = pd.read_csv(DATA_PATH, on_bad_lines='skip')
print(f"Raw shape: {df.shape}")

# This dataset has ~174k exact-duplicate reviews (same user/time/text posted
# against multiple ProductIds) - drop them so sentiment counts aren't inflated.
before = len(df)
df = df.drop_duplicates(subset=['UserId', 'ProfileName', 'Time', 'Text'])
df = df.dropna(subset=['Text'])
print(f"Dropped {before - len(df)} duplicates -> {len(df)} unique reviews")

df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
df['Date'] = pd.to_datetime(df['Time'], unit='s')
df['Year'] = df['Date'].dt.year
print(f"Working sample: {df.shape}")

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'Reviews.csv'

In [ ]:
# ------------------------------------------------------------------ #
# 2. Clean text
# ------------------------------------------------------------------ #
def clean_text(text):
    """Unescape HTML entities, strip HTML tags/URLs, collapse whitespace.
    Casing & punctuation are preserved deliberately - VADER uses them
    (e.g. capitalization and '!!!' intensify sentiment)."""
    text = html.unescape(str(text))
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['CleanText'] = df['Text'].apply(clean_text)

def rating_to_label(score):
    score = int(score) # Convert score to integer
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    return 'Positive'

df['RatingLabel'] = df['Score'].apply(rating_to_label)

In [ ]:
# ------------------------------------------------------------------ #
# 3. VADER sentiment classification
# ------------------------------------------------------------------ #
print("Classifying sentiment with VADER...")
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    s = analyzer.polarity_scores(text)
    c = s['compound']
    label = 'Positive' if c >= 0.05 else 'Negative' if c <= -0.05 else 'Neutral'
    return pd.Series([c, label])

df[['Compound', 'VaderLabel']] = df['CleanText'].apply(vader_label)

print("\nSentiment distribution (VADER, text-based):")
print((df['VaderLabel'].value_counts(normalize=True) * 100).round(1))


In [ ]:
# ------------------------------------------------------------------ #
# 4. Validate against star-rating labels
# ------------------------------------------------------------------ #
acc = accuracy_score(df['RatingLabel'], df['VaderLabel'])
print(f"\nAgreement with star-rating label: {acc*100:.1f}%")
print(classification_report(df['RatingLabel'], df['VaderLabel']))


In [ ]:
# ------------------------------------------------------------------ #
# 5. Emotion detection (NRC lexicon)
# ------------------------------------------------------------------ #
print("Detecting emotions with NRC lexicon (few minutes)...")

# Download necessary NLTK corpora for textblob/nrclex
!python -m textblob.download_corpora

def get_emotions(text):
    obj = NRCLex(None)
    obj.load_raw_text(text)
    freqs = obj.affect_frequencies
    return pd.Series([freqs.get(e, 0.0) for e in EMOTIONS])

df[EMOTIONS] = df['CleanText'].apply(get_emotions)
df['DominantEmotion'] = df[EMOTIONS].idxmax(axis=1)
df.loc[df[EMOTIONS].sum(axis=1) == 0, 'DominantEmotion'] = 'none_detected'

print("\nOverall emotion prevalence:")
print(df[EMOTIONS].mean().sort_values(ascending=False).round(4))

In [ ]:
import pandas as pd
# ------------------------------------------------------------------ #
# 6. Trends
# ------------------------------------------------------------------ #
trend = df[df['Year'] >= 2004].groupby('Year')['VaderLabel'] \
          .value_counts(normalize=True).unstack().fillna(0) * 100

# Convert 'HelpfulnessDenominator' to numeric, coercing errors to NaN
df['HelpfulnessDenominator'] = pd.to_numeric(df['HelpfulnessDenominator'], errors='coerce')
# Drop rows where 'HelpfulnessDenominator' is NaN after conversion
helpful = df.dropna(subset=['HelpfulnessDenominator'])
helpful = helpful[helpful['HelpfulnessDenominator'] >= 5]

print(f"\n% negative among highly-voted 'helpful' reviews (n={len(helpful)}): "
      f"{(helpful['VaderLabel']=='Negative').mean()*100:.1f}%  "
      f"vs overall sample: {(df['VaderLabel']=='Negative').mean()*100:.1f}%")

In [ ]:
# ------------------------------------------------------------------ #
# 7. Visualizations
# ------------------------------------------------------------------ #
print("Saving charts...")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, title in [(axes[0], 'VaderLabel', 'Sentiment from Review Text (VADER)'),
                        (axes[1], 'RatingLabel', 'Sentiment from Star Rating (reference)')]:
    vc = df[col].value_counts().reindex(ORDER)
    ax.bar(vc.index, vc.values, color=[COLORS[x] for x in vc.index])
    ax.set_title(title)
    for i, v in enumerate(vc.values):
        ax.text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_sentiment_distribution.png', dpi=150)
plt.close()

cm = confusion_matrix(df['RatingLabel'], df['VaderLabel'], labels=ORDER)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ORDER, yticklabels=ORDER)
plt.xlabel('VADER (text) label'); plt.ylabel('Star-rating label')
plt.title('Text Sentiment vs Star Rating')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/02_confusion_matrix.png', dpi=150)
plt.close()

emo_means = df[EMOTIONS].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 4.5))
sns.barplot(x=emo_means.values, y=emo_means.index, hue=emo_means.index, palette='viridis', legend=False)
plt.xlabel('Avg. frequency across reviews'); plt.title('Emotion Prevalence (NRC Lexicon)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_emotion_prevalence.png', dpi=150)
plt.close()

df.groupby('VaderLabel')[EMOTIONS].mean().reindex(ORDER).plot(
    kind='bar', stacked=True, figsize=(9, 5), colormap='tab10')
plt.title('Emotion Composition by Sentiment Class'); plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_emotion_by_sentiment.png', dpi=150)
plt.close()

plt.figure(figsize=(9, 4.5))
for label in ORDER:
    if label in trend.columns:
        plt.plot(trend.index, trend[label], marker='o', label=label, color=COLORS[label])
plt.xlabel('Year'); plt.ylabel('% of reviews'); plt.title('Sentiment Trend Over Time (2004-2012)')
plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/05_sentiment_trend.png', dpi=150)
plt.close()

extra_stop = {'product', 'flavor', 'flavour', 'food', 'br', 'one', 'amazon', 'will', 'taste'}
stopwords = STOPWORDS.union(extra_stop)
pos_text = ' '.join(df[df['VaderLabel'] == 'Positive']['CleanText']
                     .sample(min(8000, (df['VaderLabel'] == 'Positive').sum()), random_state=1))
neg_text = ' '.join(df[df['VaderLabel'] == 'Negative']['CleanText'])

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(WordCloud(width=700, height=500, background_color='white', colormap='Greens',
                          stopwords=stopwords).generate(pos_text), interpolation='bilinear')
axes[0].set_title('Positive Reviews - Most Common Words'); axes[0].axis('off')
axes[1].imshow(WordCloud(width=700, height=500, background_color='white', colormap='Reds',
                          stopwords=stopwords).generate(neg_text), interpolation='bilinear')
axes[1].set_title('Negative Reviews - Most Common Words'); axes[1].axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/06_wordclouds.png', dpi=150)
plt.close()


In [ ]:
# ------------------------------------------------------------------ #
# 8. Export labeled dataset
# ------------------------------------------------------------------ #
export_cols = ['Id', 'ProductId', 'Score', 'RatingLabel', 'Date', 'Summary', 'Text',
               'Compound', 'VaderLabel', 'DominantEmotion'] + EMOTIONS

# Convert 'Id' column to numeric, coercing errors to NaN
df['Id'] = pd.to_numeric(df['Id'], errors='coerce')
# Drop rows where 'Id' could not be converted to a number (was a non-numeric string)
df.dropna(subset=['Id'], inplace=True)
# Convert 'Id' to integer type after handling NaNs
df['Id'] = df['Id'].astype(int)

df[export_cols].sort_values('Id').to_csv('sentiment_results.csv', index=False)
print(f"\nDone. Exported {len(df)} labeled reviews to sentiment_results.csv")
print(f"Charts saved to ./{OUT_DIR}/")